In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 57. Week 39 — Data systems, bitemporal records, and PIT joins

## 学習目標

- observation/release/revision/availability/decision timeを分離できる
- future revisionを除外するPIT joinをpandasとSQLで実装できる
- row/column storage、partition、predicate pushdownのcostを説明できる
- additive schema evolutionとbreaking changeを監査できる

## 前提知識

- SQL join/window function
- timezone-aware timestamp
- SEC M6 availability contract

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 57


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

## 1. Five-time contract

| time | 意味 | 例 |
|---|---|---|
| observation | 経済量が対応する時点 | quarter end |
| release | sourceが公表した時点 | filing acceptance |
| revision | 値のversionが作られた時点 | amendment |
| availability | 保守的にpipelineで使用可能 | next business day |
| decision | modelがfeatureを読む時点 | forecast origin |

PIT joinのeligible条件は少なくとも (availability\le decision) であり、同じobservationのfuture revisionを選ばない。

In [3]:
import pandas as pd


def utc(value):
    return pd.Timestamp(value, tz="UTC")


records = (
    qt.TemporalRecord("issuer-a", "assets", 100.0, utc("2020-03-31"), utc("2020-05-01"), utc("2020-05-01"), utc("2020-05-04")),
    qt.TemporalRecord("issuer-a", "assets", 110.0, utc("2020-06-30"), utc("2020-08-01"), utc("2020-08-01"), utc("2020-08-03")),
    qt.TemporalRecord("issuer-a", "assets", 112.0, utc("2020-06-30"), utc("2020-08-01"), utc("2020-09-10"), utc("2020-09-11")),
    qt.TemporalRecord("issuer-a", "liabilities", 70.0, utc("2020-06-30"), utc("2020-08-01"), utc("2020-08-01"), utc("2020-08-03")),
)
decisions = pd.DataFrame(
    {
        "decision_id": ["early", "late"],
        "entity_id": ["issuer-a", "issuer-a"],
        "decision_time": [utc("2020-08-10"), utc("2020-09-20")],
    }
)
pandas_snapshot = qt.point_in_time_snapshot(records, decisions)
sqlite_snapshot = qt.point_in_time_snapshot_sqlite(records, decisions)
pd.testing.assert_frame_equal(pandas_snapshot, sqlite_snapshot, check_dtype=False)
assert pandas_snapshot.loc[(pandas_snapshot["decision_id"] == "early") & (pandas_snapshot["field"] == "assets"), "value"].item() == 110.0
assert pandas_snapshot.loc[(pandas_snapshot["decision_id"] == "late") & (pandas_snapshot["field"] == "assets"), "value"].item() == 112.0
display(pandas_snapshot)

,decision_id,entity_id,decision_time,field,value,observation_time,release_time,revision_time,availability_time
0,early,issuer-a,2020-08-10 00:00:00+00:00,assets,110.0,2020-06-30 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-03 00:00:00+00:00
1,early,issuer-a,2020-08-10 00:00:00+00:00,liabilities,70.0,2020-06-30 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-03 00:00:00+00:00
2,late,issuer-a,2020-09-20 00:00:00+00:00,assets,112.0,2020-06-30 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-09-10 00:00:00+00:00,2020-09-11 00:00:00+00:00
3,late,issuer-a,2020-09-20 00:00:00+00:00,liabilities,70.0,2020-06-30 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-01 00:00:00+00:00,2020-08-03 00:00:00+00:00


In [4]:
timeline = pd.DataFrame(
    [
        {"label": "initial value available", "time": utc("2020-08-03"), "value": 110.0},
        {"label": "early decision", "time": utc("2020-08-10"), "value": 110.0},
        {"label": "revision available", "time": utc("2020-09-11"), "value": 112.0},
        {"label": "late decision", "time": utc("2020-09-20"), "value": 112.0},
    ]
)
fig = go.Figure()
fig.add_scatter(x=timeline["time"], y=timeline["value"], mode="lines+markers+text", text=timeline["label"], textposition="top center")
fig.update_layout(title="Availability-time join excludes the future revision", yaxis_title="Assets value", template="plotly_white")
fig.show()

In [5]:
fixture = qt.load_sec_teaching_fixture()
train_mask = fixture.training_mask
validation_mask = fixture.validation_mask
assert train_mask.sum() == 192
assert validation_mask.sum() == 64
assert not np.any(fixture.target_available_dates >= np.datetime64("2023-10-23"))
print("fixture rows:", fixture.targets.size)
print("inner train / validation:", int(train_mask.sum()), int(validation_mask.sum()))
print("locked outer rows present: False")

fixture rows: 256
inner train / validation: 192 64
locked outer rows present: False


## 2. Columnar memory、partition、schema

Arrowはin-memory columnar layout、Parquetはcolumnar file format、DuckDBはanalytical SQL engineであり同義ではない。Coreは依存追加なしでpandas column memoryとPython row representationを比較し、SQLite window queryでSQL semanticsを検証する。production-scale Parquet/DuckDB adapterはAdvancedで、schema/hash/predicate benchmarkを伴って追加する。

In [6]:
fixture_frame = pd.DataFrame(fixture.numeric_features, columns=fixture.numeric_feature_names)
fixture_frame["partition"] = fixture.partitions
memory_audit = qt.audit_columnar_memory(fixture_frame)
schema_ok = qt.audit_schema_evolution(
    {"entity_id": "string", "value": "float64"},
    {"entity_id": "string", "value": "float64", "availability": "timestamp[UTC]"},
)
schema_bad = qt.audit_schema_evolution(
    {"entity_id": "string", "value": "float64"},
    {"entity_id": "int64"},
)
assert schema_ok.compatible
assert not schema_bad.compatible
display(pd.DataFrame([{"representation": "pandas columns", "bytes": memory_audit.total_columnar_bytes}, {"representation": "Python row dictionaries", "bytes": memory_audit.row_dictionary_bytes}]))
print("partition counts:", fixture_frame["partition"].value_counts().to_dict())
print("additive fields:", schema_ok.added_fields)
print("breaking changes:", schema_bad.removed_fields, schema_bad.changed_types)

,representation,bytes
0,pandas columns,29760
1,Python row dictionaries,96680


partition counts: {'inner_train': 192, 'inner_validation': 64}
additive fields: ('availability',)
breaking changes: ('value',) (('entity_id', 'string', 'int64'),)


`row_dictionary_bytes`はkey/valueをUTF-8文字列化した教材用下限推定で、Python object headerやallocator overheadを完全には測らない。Arrow/Parquetの実memory/file size比較へ読み替えない。

## 3. 失敗モード

- period endをavailability timeとして使う
- future revisionをlatest valueとして過去decisionへbackfillする
- Arrow/Parquet/DuckDBを同じstorage layerと呼ぶ
- schema field removalをsilent nullへ変える
- partition keyをquery patternなしで増やす

## 4. 段階別演習

### 基礎

1. five-time contractをSEC Assets factへ対応付けよ。
2. early decisionが110を選ぶSQL predicateを説明せよ。

### 標準

3. pandas/SQLite結果のindependent agreement testを書け。
4. additive/breaking schema transitionを各1件作れ。

### 研究

5. DuckDB+Parquet adapterのdependency、schema、partition、benchmark ADRを書け。

## 5. Exit Criteria

- [ ] five timestampsを分離した
- [ ] future revisionを除外した
- [ ] pandasとSQL PIT joinを照合した
- [ ] Arrow/Parquet/DuckDBの責務を区別した
- [ ] breaking schema changeをfail-closedにした

## 6. 出典

- [Apache Arrow columnar format specification](https://arrow.apache.org/docs/format/Columnar.html)
- [Apache Parquet format](https://parquet.apache.org/docs/file-format/)
- [DuckDB documentation](https://duckdb.org/docs/stable/)
- [SQLite window functions](https://www.sqlite.org/windowfunctions.html)